In [1]:
import pandas as pd
import os

# relative directory path
directory = './data/demand'
dfs = [] 

for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        # Create the full file path and read the file
        file_path = os.path.join(directory, filename)
        df_temp = pd.read_csv(file_path)
        
        # FIX THE FIRST ROW IN THE LOOP
        df_temp['SETTLEMENTDATE'] = pd.to_datetime(df_temp['SETTLEMENTDATE'])
        if not df_temp.empty:
            df_temp.loc[0, 'SETTLEMENTDATE'] -= pd.Timedelta(minutes=5)
        
        # Check if column exists to prevent errors
        print(f"File: {filename}")
        dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)
df.drop(columns=['REGION', 'PERIODTYPE'], inplace=True)
df.rename(columns={'SETTLEMENTDATE': 'DATE_TIME', 'TOTALDEMAND': 'TOTAL_DEMAND'}, inplace=True, errors='ignore')

df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'])
df = df[df['DATE_TIME'].dt.minute.isin([0, 10, 20, 30, 40, 50])].copy()
df = df.sort_values('DATE_TIME').reset_index(drop=True)


File: PRICE_AND_DEMAND_202201_NSW1.csv
File: PRICE_AND_DEMAND_202301_NSW1.csv
File: PRICE_AND_DEMAND_202401_NSW1.csv
File: PRICE_AND_DEMAND_202501_NSW1.csv
File: PRICE_AND_DEMAND_202601_NSW1.csv


In [2]:
print(df.shape)
display(df.head())
display(df.tail())
display(df.describe())

(22325, 3)


,DATE_TIME,TOTAL_DEMAND,RRP
0,2022-01-01 00:00:00,7206.03,124.86
1,2022-01-01 00:10:00,7174.26,126.02
2,2022-01-01 00:20:00,7065.84,113.54
3,2022-01-01 00:30:00,6988.47,124.06
4,2022-01-01 00:40:00,6974.20,126.24


,DATE_TIME,TOTAL_DEMAND,RRP
22320,2026-01-31 23:20:00,8556.67,63.04
22321,2026-01-31 23:30:00,8513.63,57.35
22322,2026-01-31 23:40:00,8439.73,57.35
22323,2026-01-31 23:50:00,8364.40,57.31
22324,2026-02-01 00:00:00,8258.07,57.35


,DATE_TIME,TOTAL_DEMAND,RRP
count,22325,22325.000000,22325.000000
mean,2024-01-16 21:36:00,7438.527063,80.777907
min,2022-01-01 00:00:00,4050.180000,-999.990000
25%,2023-01-08 18:00:00,6550.260000,52.970000
50%,2024-01-16 12:00:00,7277.620000,73.050000
75%,2025-01-24 06:00:00,8115.360000,92.010000
max,2026-02-01 00:00:00,13137.470000,17500.000000
std,NaN,1264.440019,300.581010


In [3]:
## CREATING COLUMNS FOR SCHOOL_HOLIDAY, PUBLIC_HOLIDAY
import holidays

# Convert to Pandas datetime for better parsing
df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'])

nsw_holidays = holidays.Australia(subdiv='NSW', years=range(2021, 2027))
df['PUBLIC_HOLIDAY'] = df['DATE_TIME'].dt.date.isin(nsw_holidays)

school_holiday_ranges = [
    # 2022
    ('2022-01-01', '2022-01-27'), # Summer 21-22 end
    ('2022-04-11', '2022-04-22'), # Autumn
    ('2022-07-04', '2022-07-15'), # Winter
    ('2022-09-26', '2022-10-07'), # Spring
    ('2022-12-21', '2022-12-31'), # Summer 22-23 start

    # 2023
    ('2023-01-01', '2023-01-26'), # Summer 22-23
    ('2023-04-10', '2023-04-21'), # Autumn
    ('2023-07-03', '2023-07-14'), # Winter
    ('2023-09-25', '2023-10-06'), # Spring
    ('2023-12-20', '2023-12-31'), # Summer 23-24 start

    # 2024
    ('2024-01-01', '2024-01-29'), # Summer 23-24 end
    ('2024-04-15', '2024-04-26'), # Autumn
    ('2024-07-08', '2024-07-19'), # Winter
    ('2024-09-30', '2024-10-11'), # Spring
    ('2023-12-23', '2023-12-31'), # Summer 24-25 start

    # 2025
    ('2025-01-01', '2025-01-30'), # Summer 24-25 end
    ('2025-04-14', '2025-04-24'), # Autumn
    ('2025-07-07', '2025-07-18'), # Winter
    ('2025-09-29', '2025-10-10'), # Spring
    ('2025-12-22', '2025-12-31'), # Summer 25-26 start

    # 2026
    ('2026-01-01', '2026-01-26'), # Summer 25-26 end
    ('2026-04-07', '2026-04-17'), # Autumn
    ('2026-07-06', '2026-07-17'), # Winter
    ('2026-09-28', '2026-10-09'), # Spring
    ('2026-12-18', '2026-12-31'), # Summer 26-27 start
]

holiday_dates = set()
for start, end in school_holiday_ranges:
    holiday_dates.update(pd.date_range(start, end).date)

df['SCHOOL_HOLIDAY'] = df['DATE_TIME'].dt.date.isin(holiday_dates)

df.head()

,DATE_TIME,TOTAL_DEMAND,RRP,PUBLIC_HOLIDAY,SCHOOL_HOLIDAY
0,2022-01-01 00:00:00,7206.03,124.86,True,True
1,2022-01-01 00:10:00,7174.26,126.02,True,True
2,2022-01-01 00:20:00,7065.84,113.54,True,True
3,2022-01-01 00:30:00,6988.47,124.06,True,True
4,2022-01-01 00:40:00,6974.20,126.24,True,True


In [4]:
import pandas as pd
import datetime

def fetch_sydney_weather(start_year, end_year):
    # Sydney Airport Station ID is usually 'YSSY' in the ASOS network
    station = 'YSSY' 
    start_date = f"{start_year}-01-01+00:00"
    end_date = f"{end_year}-12-31+23:59"
    
    # Web scraping the global archive - added 'feel' and 'p01m' for transpiration context
    url = (f"https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
           f"station={station}&data=tmpc&data=dwpc&data=relh&data=feel&data=sknt&"
           f"year1={start_year}&month1=1&day1=1&"
           f"year2={end_year}&month2=12&day2=31&"
           f"tz=Etc%2FGMT-10&format=onlycomma&latlon=no&missing=M")
    
    print("Fetching weather data")
    weather_df = pd.read_csv(url, skiprows=0, na_values='M')
    
    # Rename columns for clarity
    weather_df = weather_df.rename(columns={
        'valid': 'DATE_TIME',
        'tmpc': 'TEMPERATURE',
        'relh': 'HUMIDITY',
        'dwpc': 'DWPC',
        'feel': 'APPARENT_TEMP',
        'p01m': 'PRECIPITATION',
        'sknt': 'WIND_SPEED'
    })
    
    weather_df['DATE_TIME'] = pd.to_datetime(weather_df['DATE_TIME'])
    weather_df['APPARENT_TEMP'] = (weather_df['APPARENT_TEMP'] - 32) * 5/9 # Farenheit to Celcius
    weather_df['WIND_SPEED'] = weather_df['WIND_SPEED'] * 1.852 # Convert Wind Speed from knots to km/h (1 knot ≈ 1.852 km/h)
    
    return weather_df

weather_df = fetch_sydney_weather(2022, 2026)
weather_df.drop('station', axis=1, inplace=True, errors='ignore')
display(weather_df.head())


Fetching weather data


,DATE_TIME,TEMPERATURE,DWPC,HUMIDITY,APPARENT_TEMP,WIND_SPEED
0,2022-01-01 00:00:00,23.0,16.0,64.69,23.0,27.780
1,2022-01-01 00:30:00,23.0,16.0,64.69,23.0,22.224
2,2022-01-01 01:00:00,23.0,16.0,64.69,23.0,22.224
3,2022-01-01 01:30:00,23.0,16.0,64.69,23.0,16.668
4,2022-01-01 02:00:00,23.0,16.0,64.69,23.0,22.224


In [5]:
if 'DATE_TIME' in df.columns:
    demand_df = df.set_index('DATE_TIME').sort_index()
else:
    demand_df = df.sort_index()

if 'DATE_TIME' in weather_df.columns:
    weather_df = weather_df.set_index('DATE_TIME').sort_index()
else:
    weather_df = weather_df.sort_index()

weather_numeric = weather_df.select_dtypes(include=['number', 'float', 'int'])
weather_5min = weather_numeric.reindex(demand_df.index)

# Interpolate the gaps (Linear interpolation)
weather_5min = weather_5min.interpolate(method='linear')
weather_5min = weather_5min.bfill().ffill()

# merge back together
demand_df = demand_df.drop(columns=weather_5min.columns.intersection(demand_df.columns))
df = pd.concat([demand_df, weather_5min], axis=1).reset_index()

if 'index' in df.columns:
    df = df.rename(columns={'index': 'DATE_TIME'})
df.head()

,DATE_TIME,TOTAL_DEMAND,RRP,PUBLIC_HOLIDAY,SCHOOL_HOLIDAY,TEMPERATURE,DWPC,HUMIDITY,APPARENT_TEMP,WIND_SPEED
0,2022-01-01 00:00:00,7206.03,124.86,True,True,23.0,16.0,64.69,23.0,27.780
1,2022-01-01 00:10:00,7174.26,126.02,True,True,23.0,16.0,64.69,23.0,25.928
2,2022-01-01 00:20:00,7065.84,113.54,True,True,23.0,16.0,64.69,23.0,24.076
3,2022-01-01 00:30:00,6988.47,124.06,True,True,23.0,16.0,64.69,23.0,22.224
4,2022-01-01 00:40:00,6974.20,126.24,True,True,23.0,16.0,64.69,23.0,22.224


In [6]:
df.DATE_TIME.max()

Timestamp('2026-02-01 00:00:00')

In [7]:
### Consumer spending and confidence


## RBA Monthly Cash Rate
cash_rate_df = pd.read_csv('./data/rba_monthly_cash_rate.csv')
df['Month_Year'] = df['DATE_TIME'].dt.strftime('%Y-%m') # parse date format like '2023-01'
df = df.merge(cash_rate_df, left_on='Month_Year', right_on='DATE', how='left') # broadcast join
df = df.drop(columns=['Month_Year', 'DATE']) # clean columns

## Monthly Household Spending Indicator (MHSI)
spending_df = pd.read_csv('./data/spending_millions.csv')
df['join_key'] = df['DATE_TIME'].dt.strftime('%b-%y')

# Merge and clean up
df = df.merge(spending_df, left_on='join_key', right_on='Month', how='left')
df = df.drop(columns=['join_key', 'Month'])

# Consumer Confidence (Westpac)
consumer_conf = pd.read_csv('./data/consumer_conf.csv')
df['month_key'] = df['DATE_TIME'].dt.strftime('%Y-%m')
df = df.merge(consumer_conf, left_on='month_key', right_on='year-month', how='left')
df = df.drop(columns=['month_key', 'year-month'])

df.head(10)

,DATE_TIME,TOTAL_DEMAND,RRP,PUBLIC_HOLIDAY,SCHOOL_HOLIDAY,TEMPERATURE,DWPC,HUMIDITY,APPARENT_TEMP,WIND_SPEED,CASH_RATE,HOUSEHOLD_SPEND,SPEND_GROWTH,CONF_IDX
0,2022-01-01 00:00:00,7206.03,124.86,True,True,23.0,16.0,64.69,23.0,27.780,0.1,18099.6,-16.0,101.4
1,2022-01-01 00:10:00,7174.26,126.02,True,True,23.0,16.0,64.69,23.0,25.928,0.1,18099.6,-16.0,101.4
2,2022-01-01 00:20:00,7065.84,113.54,True,True,23.0,16.0,64.69,23.0,24.076,0.1,18099.6,-16.0,101.4
3,2022-01-01 00:30:00,6988.47,124.06,True,True,23.0,16.0,64.69,23.0,22.224,0.1,18099.6,-16.0,101.4
4,2022-01-01 00:40:00,6974.20,126.24,True,True,23.0,16.0,64.69,23.0,22.224,0.1,18099.6,-16.0,101.4
5,2022-01-01 00:50:00,6828.85,120.00,True,True,23.0,16.0,64.69,23.0,22.224,0.1,18099.6,-16.0,101.4
6,2022-01-01 01:00:00,6749.21,93.63,True,True,23.0,16.0,64.69,23.0,22.224,0.1,18099.6,-16.0,101.4
7,2022-01-01 01:10:00,6655.52,107.80,True,True,23.0,16.0,64.69,23.0,20.372,0.1,18099.6,-16.0,101.4
8,2022-01-01 01:20:00,6520.86,107.80,True,True,23.0,16.0,64.69,23.0,18.520,0.1,18099.6,-16.0,101.4
9,2022-01-01 01:30:00,6429.13,91.39,True,True,23.0,16.0,64.69,23.0,16.668,0.1,18099.6,-16.0,101.4


In [8]:
df.to_csv('./data/dataset.csv')